In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.lines import Line2D
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

In [ ]:
# Rotating magnetic field in 3-phase winding

# sampling data
samples = 90
thetas = np.linspace(0, 2*np.pi, 10*samples, endpoint=False)

# plot initialisation
plt.style.use("dark_background")
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

# plot formatting
for a in ax:
    a.get_xaxis().set_visible(False)
    a.get_yaxis().set_visible(False)

fig.subplots_adjust(
    left=0.02, right=0.98,
    bottom=0.08, top=0.85,
    wspace=0.05, hspace=0
)
fig.suptitle(
    "Rotating Magnetic Field in 3-Phase Electrical Machine", fontsize=35, y=0.97
)
ax[0].set_title("3-Phase Current Signals", fontsize=18, y=-0.1)
ax[1].set_title("Stator Windings and Magnetic Fields", fontsize=18, y=-0.1)
ax[2].set_title("Resultant Magnetic Field Vector", fontsize=18, y=-0.1)

# static plot element angles
plot_angles = np.linspace(0, 2*np.pi, 200, endpoint=True)

offsets = [n * 2*np.pi / 3 for n in range(3)]
colours = ["red", "yellow", "blue"]

# 3-phase signals
ax[0].set_xlim(0, 2*np.pi)
signal_times = []
for offset, colour in zip(offsets, colours):
    ax[0].plot(
        plot_angles, np.sin(plot_angles + offset), linewidth=3, color=colour
    )
    signal, = ax[0].plot([], [], marker="o", ms=10, color=colour)
    signal_times.append(signal)

# stator dimensions and magnetic field magnitude
for a in (ax[1], ax[2]):
    a.set_aspect("equal", adjustable="datalim")
    a.plot(
        1.5*np.cos(plot_angles), 1.5*np.sin(plot_angles), 
        linestyle="--", color="white", alpha=0.5, zorder=-10
    )
    a.scatter([0], [0], s=100, color="grey", zorder=10)

# 3-phase stator windings
centres = []
for offset, colour in zip(offsets, colours):
    sin = 1.5 * np.sin(offset)
    cos = 1.5 * np.cos(offset)
    centres.append([sin, cos])
    centres.append([-sin, -cos])
    for i in [-2, -1]:
        ax[1].add_patch(
            Circle(xy=centres[i], radius=0.12, fill=True, color=colour)
        )

# individual phase magnetic fields
phases1 = []
phases2 = []
args = [0, 0, 0, 0]
kwargs = {"scale": 1, "scale_units": "xy", "angles": "xy", "width": 0.02}
for colour in colours:
    for a, phases in ([ax[1], phases1], [ax[2], phases2]):
        phases.append(a.quiver(*args, **kwargs, color=colour))

# resultant magnetic field
resultant = ax[2].quiver(*args, **kwargs, color="white")

def update(frame):

    index = 10 * frame
    
    # current angle
    theta = thetas[index]

    # remove field lines
    for patch in list(ax[1].patches[6:] + ax[1].lines[1:]):
        patch.remove()

    # initial ax[2] vector start point
    tail = [0, 0]

    for i in range(3):

        # signal magntiude
        signal = np.sin(thetas + offsets[i])

        # update signal position
        signal_times[i].set_data([theta], [signal[index]])

        R = 0.8 * signal[index]

        # loop over each stator winding
        for j in range(2):

            # ensure each widning has opposite current direction
            R *= (-1)**j

            loc = centres[2*i+j]

            # update field lines
            r = abs(R)
            while r > 0:
                field_line = Circle(
                    xy=loc, radius=r, fill=False, alpha=0.5, zorder=-10
                )
                r -= 0.2
                ax[1].add_patch(field_line)

            # update current direction symbols
            if R > 0: # out of page
                size = R / 10
                current = Circle(xy=loc, radius=size, fill=True, color="black")
                ax[1].add_patch(current)
            if R < 0: # into page
                size = R / 15
                current = Line2D(
                    [loc[0]-size, loc[0]+size], 
                    [loc[1]-size, loc[1]+size], 
                    linewidth=3, color="black"
                )
                ax[1].add_line(current)
                current = Line2D(
                    [loc[0]-size, loc[0]+size], 
                    [loc[1]+size, loc[1]-size], 
                    linewidth=3, color="black"
                )
                ax[1].add_line(current)

        # magnetic field vectors
        x = signal[index] * np.cos(offsets[i])
        y = signal[index] * np.sin(-offsets[i])

        # update vectors
        phases1[i].set_UVC(x, y)
        phases2[i].set_offsets([tail])
        phases2[i].set_UVC(x, y)

        # update next vector starting point
        tail[0] += x
        tail[1] += y

    # update resultant magnetic field vector
    resultant.set_offsets([0, 0])
    resultant.set_UVC(*tail)

    return [*signal_times, *phases1, *phases2, resultant]

# animate rotation
anim = FuncAnimation(fig, update, frames=samples, blit=True)
anim.save("rotating-magnetic-field.gif", writer=PillowWriter(fps=18))
plt.close(fig)
HTML(anim.to_jshtml())